In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


In [6]:
TRAIN_FILE = "/content/dataset-1.csv"


In [7]:
TARGET = "Churn"
DROP_COLUMNS = []


In [8]:
df = pd.read_csv(TRAIN_FILE)

In [9]:
print(df.head())
print("Rows, columns:", df.shape)
df.info()

print("\nMissing values:")
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())


   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  4950-BDEUX    Male              0      No         No      35           No   
1  7993-NQLJE    Male              0     Yes        Yes      15          Yes   
2  7321-ZNSLA    Male              0     Yes        Yes      13           No   
3  4922-CVPDX  Female              0     Yes         No      26          Yes   
4  2903-YYTBW    Male              0     Yes        Yes       1          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...              Yes   
1                No     Fiber optic            Yes  ...               No   
2  No phone service             DSL            Yes  ...               No   
3                No             DSL             No  ...              Yes   
4                No             DSL             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [10]:
X = df.drop(columns=[TARGET] + DROP_COLUMNS)
y = df[TARGET]

print("\nClass counts:")
print(y.value_counts())



Class counts:
Churn
False    4139
True     1495
Name: count, dtype: int64


In [11]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


In [12]:
numeric_columns = X.select_dtypes(include="number").columns.tolist()
categorical_columns = X.select_dtypes(exclude="number").columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_columns),
    ("categorical", categorical_pipeline, categorical_columns)
])


In [13]:
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])


In [14]:
model.fit(X_train, y_train)
predictions = model.predict(X_valid)


In [15]:
print("\nAccuracy:", accuracy_score(y_valid, predictions))

print("\nClassification report:")
print(classification_report(
    y_valid,
    predictions,
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_valid, predictions))

comparison = pd.DataFrame({
    "Actual": y_valid.to_numpy(),
    "Predicted": predictions
})

print(comparison.head(10))



Accuracy: 0.7941437444543035

Classification report:
              precision    recall  f1-score   support

       False       0.82      0.92      0.87       828
        True       0.67      0.44      0.53       299

    accuracy                           0.79      1127
   macro avg       0.75      0.68      0.70      1127
weighted avg       0.78      0.79      0.78      1127

Confusion matrix:
[[763  65]
 [167 132]]
   Actual  Predicted
0   False      False
1   False      False
2   False      False
3   False      False
4   False      False
5   False      False
6   False      False
7   False      False
8   False      False
9   False      False


In [16]:
TEST_FILE = "/content/dataset-1.csv"
test_df = pd.read_csv(TEST_FILE)
X_test = test_df[X.columns]


In [17]:
model.fit(X, y)
test_predictions = model.predict(X_test)

submission = pd.DataFrame({
    TARGET: test_predictions
})

submission.to_csv("/content/classification_submission.csv", index=False)
print("\nClassification submission saved!")



Classification submission saved!


In [18]:
from sklearn.linear_model import LogisticRegression

In [19]:
("classifier", LogisticRegression(max_iter=2000))

('classifier', LogisticRegression(max_iter=2000))